# Qwen3.5-0.8B V4: Full attention FIRST → Progressive Delta SECOND

This notebook produces **two distinct models**: full-attention-only (6 replacements, retaining all 18 original DeltaNet layers) and an experimental progressively substituted Delta version (18 possible replacements, selected with rollback). Results and models are saved separately.

**Why:** V3 recorded ~59.4% holdout next-token teacher agreement despite faster prefill and exhibited repeating chat output. Replacing 18 DeltaNet layers before validating the six full attention layers obscured the source of the problem. V4 checks full-attention parity first, then ranks individual Delta layers on TRAIN data, adds them gradually, trains and rolls back groups that fail DEV fidelity.

**Splits:** official WikiText train/validation/test; holdout is sealed until final evaluation. FFNs/norms remain frozen. No CeNN. Text-only. use_cache=False; no streaming performance claims.

**Output:** full-only checkpoint, best accepted Delta checkpoint (or first rejected Delta diagnostic-only checkpoint), statistics ZIP and optional model ZIPs; separate Hugging Face gates for each model.


In [ ]:
#@title 1. Environment and experiment parameters
%pip -q install -U "git+https://github.com/huggingface/transformers.git@main" "accelerate>=1.0" "datasets>=3" "huggingface_hub>=0.34" "safetensors>=0.5"
%pip -q install --no-deps "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@a17f294d0144c2a09c41ace8b9fbea6108af4eaa"
from pathlib import Path
from datetime import datetime, timezone
import os, random, math, json, copy, gc, hashlib, zipfile, re, shutil, time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import HfApi, login, get_token
from IPython.display import display, FileLink
from tinycenn_lm import qwen35_window_anchor_v4 as wa

MODEL_ID="Qwen/Qwen3.5-0.8B"
SEED=20261018
MAX_TOKENS=192 #@param {type:"integer"}
DELTA_WINDOW=128 #@param {type:"integer"}
FULL_WINDOW=512 #@param {type:"integer"}
ANCHORS=16 #@param {type:"integer"}
CHUNK=64 #@param {type:"integer"}
TRAIN_EXAMPLES=320 #@param {type:"integer"}
DEV_EXAMPLES=64 #@param {type:"integer"}
HOLDOUT_EXAMPLES=96 #@param {type:"integer"}
RANKING_EXAMPLES=12 #@param {type:"integer"}
DELTA_STEPS=100 #@param {type:"integer"}
GRAD_ACCUM=2 #@param {type:"integer"}
LORA_RANK=12 #@param {type:"integer"}
LR_START=0.00035 #@param {type:"number"}
EVAL_EVERY=25
DEV_MAX_DROP=0.025
AUTO_DOWNLOAD_STATS=True #@param {type:"boolean"}
AUTO_DOWNLOAD_MODELS=False #@param {type:"boolean"}
UPLOAD_IF_QUALIFIED=True #@param {type:"boolean"}
HF_PRIVATE=True #@param {type:"boolean"}
FULL_HF_NAME="qwen35-08b-v4-full6-only" #@param {type:"string"}
DELTA_HF_NAME="qwen35-08b-v4-progressive-delta" #@param {type:"string"}
BASE=Path("/content/Qwen35_V4_FullFirst_ProgressiveDelta")
REPORTS=BASE/"statistics"
FULL_EXPORT=BASE/"full6_only_model"
DELTA_EXPORT=BASE/"progressive_delta_model"
DIAGNOSTIC_EXPORT=BASE/"rejected_delta_diagnostic_model"
for path in (BASE,REPORTS,FULL_EXPORT):path.mkdir(parents=True,exist_ok=True)
assert 8<=MAX_TOKENS<=512 and FULL_WINDOW>=MAX_TOKENS
assert 0<=ANCHORS<=DELTA_WINDOW and DELTA_WINDOW>=16
assert TRAIN_EXAMPLES>=RANKING_EXAMPLES>=4
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
assert torch.cuda.is_available(),"Enable GPU in Colab runtime settings"
torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda",torch.cuda.current_device())
ATTENTION_SETTINGS={"delta_window":DELTA_WINDOW,"full_window":FULL_WINDOW,
                    "anchors":ANCHORS,"chunk_size":CHUNK}
print("GPU",torch.cuda.get_device_name(),"Sequence: FULL FIRST → DELTA SECOND")


In [ ]:
#@title 2. Three official WikiText splits: TRAIN, DEV, sealed HOLDOUT
# Unlike V2, validation/test are not slices of the training split.
DATA_SOURCE="wikitext-2-raw-v1"
def clean_texts(split,needed):
    ds=load_dataset("Salesforce/wikitext","wikitext-2-raw-v1",split=split)
    unique=[];seen=set()
    for row in ds:
        value=" ".join(str(row["text"]).split())
        if 80<=len(value)<=850 and not value.startswith("=") and value not in seen:
            unique.append(value);seen.add(value)
    if len(unique)<needed:
        raise RuntimeError(f"Split {split} has only {len(unique)} eligible rows; need {needed}")
    rng=random.Random(SEED+{"train":11,"validation":23,"test":41}[split])
    rng.shuffle(unique)
    return unique[:needed]
try:
    train_text=clean_texts("train",TRAIN_EXAMPLES)
    dev_text=clean_texts("validation",DEV_EXAMPLES)
    holdout_text=clean_texts("test",HOLDOUT_EXAMPLES)
except Exception as exc:
    print("Official data unavailable; deterministic fallback for debugging ONLY:",repr(exc))
    DATA_SOURCE="synthetic_fallback_not_valid_for_publication"
    subjects=["A scientist","A library","An engineer","A hospital","A teacher",
              "A spacecraft","A mathematician","An orchestra","A historian","A botanist"]
    verbs=["described","studied","measured","compared","discussed",
           "reconsidered","observed","evaluated","investigated"]
    suffixes=["a new method using controlled experiments and careful analysis.",
              "the importance of verification and repeating the measurement.",
              "several observations that challenged the previous explanation.",
              "the distinction between efficiency, accuracy and usefulness.",
              "how practical design choices affected the final result."]
    needed=TRAIN_EXAMPLES+DEV_EXAMPLES+HOLDOUT_EXAMPLES
    unique=[(f"{subjects[i%len(subjects)]} "
             f"{verbs[(i//len(subjects))%len(verbs)]} "
             f"{suffixes[(i//(len(subjects)*len(verbs)))%len(suffixes)]} "
             f"Deterministic synthetic debug record {i}.")
            for i in range(needed)]
    print("WARNING: synthetic data is NOT a quality benchmark and HF upload will be blocked.")
    random.Random(SEED).shuffle(unique)
    train_text=unique[:TRAIN_EXAMPLES]
    dev_text=unique[TRAIN_EXAMPLES:TRAIN_EXAMPLES+DEV_EXAMPLES]
    holdout_text=unique[TRAIN_EXAMPLES+DEV_EXAMPLES:
                        TRAIN_EXAMPLES+DEV_EXAMPLES+HOLDOUT_EXAMPLES]
assert not set(train_text)&set(dev_text)
assert not set(train_text)&set(holdout_text)
assert not set(dev_text)&set(holdout_text)
print(DATA_SOURCE,"official split sizes",len(train_text),len(dev_text),len(holdout_text))



In [ ]:
#@title 3. Official Qwen3.5 text-only teacher; cache train and DEV logits once
teacher,tokenizer,source_root=wa.load_official_text(MODEL_ID,dtype=torch.float16)
teacher.to(device).eval().requires_grad_(False)
original_config=teacher.config.to_dict()

def make_record(text):
    ids=tokenizer(text,add_special_tokens=False)["input_ids"][:MAX_TOKENS+1]
    if len(ids)<8:raise ValueError("Text too short")
    return {"input_ids":torch.tensor([ids[:-1]],dtype=torch.long),
            "gold_next":int(ids[-1]),"text":text,"length":len(ids)-1}
train_rows=[make_record(s) for s in train_text]
dev_rows=[make_record(s) for s in dev_text]
# Holdout records are NOT yet materialized or evaluated.
@torch.inference_mode()
def attach_reference(records):
    for index,rec in enumerate(records):
        x=rec["input_ids"].to(device)
        logits=teacher(input_ids=x,use_cache=False,logits_to_keep=1).logits[:, -1]
        rec["teacher_logits"]=logits.float().cpu().contiguous()
        if index%25==0:print("Reference logits",index,"/",len(records))
    return records
train_rows=attach_reference(train_rows)
dev_rows=attach_reference(dev_rows)

teacher.cpu();gc.collect();torch.cuda.empty_cache()
print('Teacher logits cached for TRAIN/DEV only; HOLDOUT untouched.')


In [ ]:
#@title 4. Verify FULL attention first; export full-only version
def frozen_digest(model):
    h=hashlib.sha256()
    for i,L in enumerate(model.model.layers):
        for field in ("mlp","input_layernorm","post_attention_layernorm"):
            for key,tensor in getattr(L,field).state_dict().items():
                h.update(f"{i}/{field}/{key}".encode())
                h.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

@torch.inference_mode()
def assess(model,records):
    model.eval()
    total=matches=0
    divergence=student_loss=teacher_loss=0.0
    for rec in records:
        ids=rec["input_ids"].to(device)
        with torch.autocast("cuda",dtype=torch.float16):
            logits=model(input_ids=ids,use_cache=False,logits_to_keep=1).logits[:,-1].float()
        ref=rec["teacher_logits"].to(device).float()
        p=logits.softmax(-1);q=ref.softmax(-1);m=(p+q)/2
        div=0.5*((p*(p.clamp_min(1e-10).log()-m.clamp_min(1e-10).log())).sum()
                 +(q*(q.clamp_min(1e-10).log()-m.clamp_min(1e-10).log())).sum())
        divergence+=float(div)
        matches+=int(logits.argmax(-1).item()==ref.argmax(-1).item())
        gold=torch.tensor([rec["gold_next"]],device=device)
        student_loss+=float(F.cross_entropy(logits,gold))
        teacher_loss+=float(F.cross_entropy(ref,gold))
        total+=1
    return {"n":total,"teacher_agreement":matches/total,"mean_js":divergence/total,
            "student_nll":student_loss/total,"teacher_nll":teacher_loss/total,
            "ppl_ratio":math.exp(min(30,(student_loss-teacher_loss)/total))}

FROZEN_DIGEST=frozen_digest(teacher)
FULL_IDS=[i for i,L in enumerate(teacher.model.layers)
          if L.block_type=="full_attention"]
DELTA_IDS=[i for i,L in enumerate(teacher.model.layers)
           if L.block_type=="linear_attention"]
assert len(FULL_IDS)==6 and len(DELTA_IDS)==18

def create_manifest(model,variant,dev,stage_accepted=False):
    installed=wa.stage_indices(model)
    assert installed["full"]==FULL_IDS,installed
    assert frozen_digest(model)==FROZEN_DIGEST,"Pretrained FFNs or block norms changed"
    return {
        "architecture":"Qwen3.5_0.8B_V4_FullFirst_ProgressiveDelta",
        "variant":variant,"source_model":MODEL_ID,"dtype":"float16",
        "attention_settings":ATTENTION_SETTINGS,
        "replaced_full_indices":installed["full"],
        "replaced_delta_indices":installed["delta"],
        "native_delta_layers_remaining":18-len(installed["delta"]),
        "text_only":True,"requires_use_cache_false":True,"no_cenn":True,
        "frozen_ffn_norm_sha256":FROZEN_DIGEST,
        "dev":dev,"stage_accepted":bool(stage_accepted),
        "dataset":DATA_SOURCE,"created_utc":datetime.now(timezone.utc).isoformat()
    }

def save_model_variant(model,folder,manifest):
    folder.mkdir(parents=True,exist_ok=True)
    model.half()
    wa.save_custom(model,tokenizer,folder,manifest)
    (folder/"README.md").write_text(
        f"# Qwen3.5 V4 — {manifest['variant']}\n\n"
        f"Full replaced {len(manifest['replaced_full_indices'])}; "
        f"Delta replaced {len(manifest['replaced_delta_indices'])}. "
        "Research only; text-only; use_cache=False. "
        "Load using tinycenn_lm.qwen35_window_anchor_v4.load_custom. "
        "Diagnostic-only variants are not qualified for deployment.\n")
    (folder/"requirements.txt").write_text(
        "torch>=2.4\nsafetensors>=0.5\n"
        "git+https://github.com/huggingface/transformers.git@main\n"
        "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@a17f294d0144c2a09c41ace8b9fbea6108af4eaa\n")
    assert (folder/"model.safetensors").is_file()
    model.to(device).eval()

full_model=copy.deepcopy(teacher)
wa.install_selected(full_model,full_indices=FULL_IDS,**ATTENTION_SETTINGS)
full_model.to(device).eval().requires_grad_(False)
assert wa.stage_indices(full_model)=={"full":FULL_IDS,"delta":[]}
assert frozen_digest(full_model)==FROZEN_DIGEST
FULL_DEV=assess(full_model,dev_rows)
FULL_PASS=bool(FULL_DEV["teacher_agreement"]>=0.99
               and FULL_DEV["mean_js"]<=0.002)
FULL_MANIFEST=create_manifest(full_model,"full6_only",FULL_DEV,FULL_PASS)
save_model_variant(full_model,FULL_EXPORT,FULL_MANIFEST)
(REPORTS/"full_only_dev.json").write_text(json.dumps(
    {"passes_full_stage_gate":FULL_PASS,"manifest":FULL_MANIFEST},indent=2))
print("Full attention first:",FULL_IDS)
print("All 18 native DeltaNet layers RETAINED.")
print("FULL-ONLY DEV",FULL_DEV,"PASS:",FULL_PASS)
print("Full-only standalone model saved:",FULL_EXPORT)


In [ ]:
#@title 5. Rank all 18 potential Delta replacements by TRAIN-only damage
# All 6 full mixers are already replaced; native DeltaNet is preserved
# except for the ONE temporarily substituted module being measured.
DELTA_RANKING=[]
if FULL_PASS:
    subset=train_rows[:RANKING_EXAMPLES]
    reference_train=assess(full_model,subset)
    for index in DELTA_IDS:
        layer=full_model.model.layers[index]
        native=layer.linear_attn
        assert not isinstance(native,wa.WindowAnchorDeltaV2)
        candidate=wa.WindowAnchorDeltaV2(
            native,DELTA_WINDOW,ANCHORS,CHUNK).to(device).eval()
        candidate.requires_grad_(False)
        layer.linear_attn=candidate
        try:
            result=assess(full_model,subset)
            DELTA_RANKING.append({
                "layer":index,"teacher_agreement":result["teacher_agreement"],
                "mean_js":result["mean_js"],"ppl_ratio":result["ppl_ratio"],
                "top1_agreement_drop":
                   reference_train["teacher_agreement"]-result["teacher_agreement"]})
        finally:
            layer.linear_attn=native
            del candidate
            gc.collect()
        print("Single Delta trial L",index,
              "agreement",round(result["teacher_agreement"],4),
              "JS",round(result["mean_js"],5))
    DELTA_RANKING.sort(key=lambda x:(
        -x["teacher_agreement"],x["mean_js"],x["ppl_ratio"]))
    RANKED_DELTA=[x["layer"] for x in DELTA_RANKING]
else:
    print("FULL-6 fidelity gate failed: progressive Delta training blocked.")
    RANKED_DELTA=[]
(REPORTS/"single_delta_rankings.json").write_text(json.dumps({
    "full_only_passed":FULL_PASS,"teacher_train_baseline":
         reference_train if FULL_PASS else None,
    "train_probe_count":RANKING_EXAMPLES,"ranked_layers":DELTA_RANKING},indent=2))
print("Candidate Delta order (least damaging first):",RANKED_DELTA)


In [ ]:
#@title 6. Train progressive Delta replacements: 1 → 3 → 6 → 12 → 18, with rollback
# DeltaNet remains original in all unselected layers. A failed group restores
# its exact native modules, without mutating any previously accepted layer.
class DeltaLoRA(nn.Module):
    def __init__(self,base,rank=LORA_RANK):
        super().__init__()
        assert isinstance(base,nn.Linear)
        self.base=base.requires_grad_(False)
        dev=base.weight.device
        self.A=nn.Parameter(torch.empty(rank,base.in_features,
                             device=dev,dtype=torch.float32))
        self.B=nn.Parameter(torch.zeros(base.out_features,rank,
                             device=dev,dtype=torch.float32))
        nn.init.normal_(self.A,std=0.01)
        self.scale=1.0/rank
    def forward(self,x):
        return self.base(x)+F.linear(F.linear(x,self.A),self.B)*self.scale
    @torch.no_grad()
    def merge(self):
        delta=(self.B@self.A)*self.scale
        self.base.weight.add_(delta.to(dtype=self.base.weight.dtype))
        return self.base

def score(m):
    return (round(m["teacher_agreement"]
                  -0.4*max(0,m["ppl_ratio"]-1)-0.7*m["mean_js"],8),
            m["teacher_agreement"],-m["mean_js"])

def stage_train(new_layer_indices,stage_number):
    modules=[full_model.model.layers[i].linear_attn for i in new_layer_indices]
    trainable=[]
    wrapper_map=[]
    for module in modules:
        assert isinstance(module,wa.WindowAnchorDeltaV2)
        module.requires_grad_(False)
        for key in ("in_proj_qkv","in_proj_b","in_proj_z","out_proj"):
            lora=DeltaLoRA(getattr(module,key))
            setattr(module,key,lora)
            wrapper_map.append((module,key,lora))
            trainable.extend((lora.A,lora.B))
        for p in (module.log_temperature,module.log_decay,
                  module.direct_mix_logit,module.output_gain,
                  module.output_bias):
            p.requires_grad_(True)
            trainable.append(p)
        # GradScaler requires FP32 parameters when unscaling gradients.
        module.norm.float().requires_grad_(True)
        trainable.extend([p for p in module.norm.parameters() if p.requires_grad])
    assert trainable and all(p.dtype==torch.float32 for p in trainable)
    optim=torch.optim.AdamW(trainable,lr=LR_START,weight_decay=0)
    scaler=torch.amp.GradScaler("cuda")
    baseline=assess(full_model,dev_rows)
    best_metric=baseline
    best_params=[p.detach().cpu().clone() for p in trainable]
    best_step=0
    train_order=list(range(len(train_rows)))
    rng=random.Random(SEED+stage_number*307)
    rng.shuffle(train_order)
    pos=0
    log=[{"step":0,"dev":baseline}]
    for step in range(1,DELTA_STEPS+1):
        progress=(step-1)/max(1,DELTA_STEPS-1)
        lr=LR_START*(0.15+0.85*(0.5+0.5*math.cos(math.pi*progress)))
        optim.param_groups[0]["lr"]=lr
        optim.zero_grad(set_to_none=True)
        losses=[]
        for _ in range(GRAD_ACCUM):
            row=train_rows[train_order[pos]]
            pos=(pos+1)%len(train_order)
            if pos==0:rng.shuffle(train_order)
            ids=row["input_ids"].to(device)
            target=row["teacher_logits"].to(device).float()
            gold=torch.tensor([row["gold_next"]],device=device)
            with torch.autocast("cuda",dtype=torch.float16):
                pred=full_model(input_ids=ids,use_cache=False,
                                logits_to_keep=1).logits[:,-1].float()
                tau=2.0
                kd=F.kl_div(F.log_softmax(pred/tau,dim=-1),
                            F.softmax(target/tau,dim=-1),
                            reduction="batchmean")*tau*tau
                loss=kd+0.03*F.cross_entropy(pred,gold)
            if not bool(torch.isfinite(loss).item()):
                raise FloatingPointError(f"Stage {stage_number}, step {step}: nonfinite loss")
            scaler.scale(loss/GRAD_ACCUM).backward()
            losses.append(float(loss.detach()))
        scaler.unscale_(optim)
        norm=torch.nn.utils.clip_grad_norm_(trainable,0.6)
        if bool(torch.isfinite(norm).item()):
            scaler.step(optim)
        else:
            print("Skipped nonfinite gradient, stage",stage_number,"step",step)
        scaler.update()
        if step%EVAL_EVERY==0 or step==1 or step==DELTA_STEPS:
            dev=assess(full_model,dev_rows)
            if score(dev)>score(best_metric):
                best_metric=dev
                best_step=step
                best_params=[p.detach().cpu().clone() for p in trainable]
            log.append({"step":step,"loss":float(np.mean(losses)),
                        "dev":dev,"best_step":best_step})
            print(f"Stage {stage_number} step {step}/{DELTA_STEPS}: "
                  f"agree={dev['teacher_agreement']:.2%}, JS={dev['mean_js']:.5f}, "
                  f"best={best_step}")
    with torch.no_grad():
        for p,checkpoint in zip(trainable,best_params):
            p.copy_(checkpoint.to(device=p.device,dtype=p.dtype))
    for module,name,lora in wrapper_map:
        assert isinstance(getattr(module,name),DeltaLoRA)
        setattr(module,name,lora.merge())
    for module in modules:
        module.requires_grad_(False)
        module.half()
    del optim,scaler,trainable,best_params
    gc.collect();torch.cuda.empty_cache()
    return {"best_step":best_step,"history":log,
            "best_dev":best_metric,"final_dev":assess(full_model,dev_rows)}

STAGE_BOUNDARIES=(0,1,3,6,12,18)
STAGE_RECORDS=[]
ACCEPTED_DELTA=[]
CURRENT_DEV=FULL_DEV
DELTA_CANDIDATE_FOLDER=None
if FULL_PASS:
    for stage in range(1,len(STAGE_BOUNDARIES)):
        first,last=STAGE_BOUNDARIES[stage-1:stage+1]
        fresh=RANKED_DELTA[first:last]
        before={i:full_model.model.layers[i].linear_attn for i in fresh}
        # NEVER modify any of the previously accepted Delta layers.
        wa.install_selected(full_model,delta_indices=fresh,
                            **ATTENTION_SETTINGS)
        full_model.eval().requires_grad_(False)
        training=stage_train(fresh,stage)
        measured=training["final_dev"]
        passes=bool(
            measured["teacher_agreement"]>=max(0.95,
                   CURRENT_DEV["teacher_agreement"]-DEV_MAX_DROP)
            and measured["mean_js"]<=0.04
            and measured["ppl_ratio"]<=1.15)
        row={"stage":stage,"nominal_delta_target":last,"new_layers":fresh,
             "previous_dev":CURRENT_DEV,"candidate_dev":measured,
             "best_step":training["best_step"],"accepted":passes,
             "selected_delta_before":list(ACCEPTED_DELTA)}
        (REPORTS/f"stage{stage}_training.json").write_text(json.dumps(training,indent=2))
        # Ensure at least one genuine Delta version is preserved even when
        # all candidate groups degrade quality. It is DIAGNOSTIC ONLY.
        if not ACCEPTED_DELTA and DELTA_CANDIDATE_FOLDER is None:
            diagnostic=create_manifest(
                full_model,"rejected_delta_diagnostic",measured,False)
            save_model_variant(full_model,DIAGNOSTIC_EXPORT,diagnostic)
            DELTA_CANDIDATE_FOLDER=DIAGNOSTIC_EXPORT
        if passes:
            ACCEPTED_DELTA.extend(fresh)
            ACCEPTED_DELTA.sort()
            CURRENT_DEV=measured
        else:
            for index,native in before.items():
                full_model.model.layers[index].linear_attn=native
            gc.collect();torch.cuda.empty_cache()
        row["selected_delta_after"]=list(ACCEPTED_DELTA)
        STAGE_RECORDS.append(row)
        assert frozen_digest(full_model)==FROZEN_DIGEST
        print(f"Stage {stage}: {'ACCEPTED' if passes else 'ROLLED BACK'}; "
              f"selected {len(ACCEPTED_DELTA)}/18 Delta layers")
    if ACCEPTED_DELTA:
        best_manifest=create_manifest(
            full_model,"progressive_delta_accepted",CURRENT_DEV,True)
        save_model_variant(full_model,DELTA_EXPORT,best_manifest)
        DELTA_CANDIDATE_FOLDER=DELTA_EXPORT
else:
    print("No Delta substitutions attempted: full-only compatibility failed.")

DELTA_STAGE_PASSED=bool(ACCEPTED_DELTA)
(REPORTS/"progressive_delta_stages.json").write_text(json.dumps({
    "full_stage_passed":FULL_PASS,"stages":STAGE_RECORDS,
    "accepted_delta_layer_indices":ACCEPTED_DELTA,
    "candidate_path":str(DELTA_CANDIDATE_FOLDER) if DELTA_CANDIDATE_FOLDER else None,
    "delta_stage_accepted":DELTA_STAGE_PASSED},indent=2))
print("Final accepted Delta layers:",ACCEPTED_DELTA)
print("Selected Delta checkpoint folder:",DELTA_CANDIDATE_FOLDER)
